# AI-06: Fake Drug Text/Barcode Checker
**Author:** Alhassan Aliyu Liman

A classifier + lookup tool that flags suspicious drug/product entries against NAFDAC's official Greenbook registry, to help fight counterfeit medicines in Nigeria.

**Pipeline:** scrape NAFDAC data -> generate synthetic fakes -> engineer features -> train a classifier -> evaluate -> build an explainable checker app.


## Step 1: Load & parse the NAFDAC Greenbook data
Raw scraped listings are messy text. We parse them into clean rows: product name, NRN registration number, category.

In [3]:
"""
STEP 1: Parse raw scraped NAFDAC Greenbook text into a clean, structured CSV.

Why this step matters:
Web-scraped data is messy — it comes as long strings mixing product name,
dosage form, active ingredient, strength, and registration number (NRN) all
together. Before we can do ANY machine learning, we need this in neat rows
and columns (like a spreadsheet). This is called "data cleaning" and it's
usually 70% of the work in a real ML project.
"""
import re
import csv

def parse_category_file(filepath, category_name):
    """
    Each line in our raw file looks like:
    [Product Name** Form Ingredient Strength NRN: A6-100070](https://..*/./details/6931)

    We use a "regular expression" (regex) - a pattern-matching tool - to pull
    out the pieces we care about: the display text, and the NRN code.
    """
    rows = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Match: [ ...text... ](url)
            m = re.match(r"^\[(.*?)\]\((https?://\S+)\)$", line)
            if not m:
                continue
            display_text, url = m.group(1), m.group(2)

            # Pull out the NRN registration number, e.g. "NRN: A6-100070"
            nrn_match = re.search(r"NRN:\s*([A-Za-z0-9\-]+)", display_text)
            nrn = nrn_match.group(1) if nrn_match else None

            # The product's short name is usually everything before the
            # first '**', '##', or '#' marker NAFDAC uses in their listing
            name_match = re.match(r"^(.*?)(\*\*|##|#)", display_text)
            if name_match:
                short_name = name_match.group(1).strip()
            else:
                # Fallback: take text up to " NA " or the first number-heavy
                # dosage chunk - just use first 8 words as a rough cut
                short_name = " ".join(display_text.split()[:8])

            rows.append({
                "product_name": short_name,
                "full_listing_text": display_text,
                "nrn": nrn,
                "category": category_name,
                "source_url": url,
            })
    return rows


if __name__ == "__main__":
    all_rows = []
    all_rows += parse_category_file("data/raw_vaccines.txt", "Vaccines and Biologics")

    out_path = "data/nafdac_products_clean.csv"
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "product_name", "full_listing_text", "nrn", "category", "source_url"
        ])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"Parsed {len(all_rows)} genuine NAFDAC products -> {out_path}")
    print("\nSample rows:")
    for r in all_rows[:5]:
        print(f" - {r['product_name']}  |  NRN: {r['nrn']}")


Parsed 156 genuine NAFDAC products -> data/nafdac_products_clean.csv

Sample rows:
 - Actemra 162 mg  |  NRN: A6-100070
 - Actemra 200 mg  |  NRN: A6-100067
 - Actemra 400 mg  |  NRN: A6-100066
 - NovoRapid FlexPen 100 units/mL Solution for Injection in Pre-Filled Pen  |  NRN: A6-0207
 - Polio Sabin Bivalent Oral Poliomyelitis Vaccine Types 1 and 3 (BOPV)  |  NRN: A6-0082


### Step 1b: Add the Drugs category (different raw format)
NAFDAC serves the Drugs category as plain run-on text with no [name](url) brackets, unlike Vaccines. We split on the one reliable anchor - the NRN code pattern - to pull out each entry, then merge it into the same clean genuine-products table.

In [4]:
"""
STEP 1b: Parse the Drugs category, which NAFDAC serves in a DIFFERENT raw
format than Vaccines did - plain text with no [name](url) brackets, just
one long run-on string of "Name FORM Ingredient Strength NRN: xxx" blocks
back to back.

The trick: NRN codes ("NRN: A4-100160") are the one reliable anchor. We
split the whole blob on that pattern - everything between one NRN and the
next NRN belongs to the entry that ends with the second NRN.
"""
import re
import csv

NRN_TOKEN = re.compile(r"NRN:\s*([A-Za-z0-9]{1,4}-\d{2,7})")

def parse_plaintext_drugs(filepath, category_name="Drugs"):
    text = open(filepath, encoding="utf-8").read()

    # Find every "...stuff... NRN: CODE" chunk
    matches = list(NRN_TOKEN.finditer(text))
    rows = []
    start = 0
    for m in matches:
        chunk = text[start:m.end()].strip()
        nrn = m.group(1)
        # The short product name is the text before the '##' / '**' / '#'
        # marker NAFDAC uses, same trick as the bracketed format
        name_match = re.match(r"^(.*?)(\*\*|##|#)", chunk)
        if name_match:
            short_name = name_match.group(1).strip()
        else:
            short_name = " ".join(chunk.split()[:6])
        rows.append({
            "product_name": short_name,
            "full_listing_text": chunk,
            "nrn": nrn,
            "category": category_name,
            "source_url": "",
        })
        start = m.end()
    return rows


if __name__ == "__main__":
    drug_rows = parse_plaintext_drugs("data/raw_drugs_plaintext.txt")
    print(f"Parsed {len(drug_rows)} genuine Drugs entries")
    for r in drug_rows[:5]:
        print(f" - {r['product_name']}  |  NRN: {r['nrn']}")

    # Merge with the existing vaccines dataset into one combined file
    existing = list(csv.DictReader(open("data/nafdac_products_clean.csv", encoding="utf-8")))
    combined = existing + drug_rows

    with open("data/nafdac_products_clean.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "product_name", "full_listing_text", "nrn", "category", "source_url"
        ])
        writer.writeheader()
        writer.writerows(combined)

    print(f"\nCombined dataset now has {len(combined)} genuine products -> data/nafdac_products_clean.csv")


Parsed 79 genuine Drugs entries
 - AG Spironolactone  |  NRN: A4-100160
 - Artheget EZ  |  NRN: A4-6238
 - Crushmal  |  NRN: A4-100161
 - Dolburg 50  |  NRN: A4-9351
 - Hanmal  |  NRN: B4-4875

Combined dataset now has 235 genuine products -> data/nafdac_products_clean.csv


## Step 2: Generate synthetic 'suspicious' examples
NAFDAC only gives us genuine products - there's no public dataset of confirmed fakes. We simulate the 3 real-world counterfeiting patterns: typo'd names, tampered NRNs, and fully fabricated products. This is standard practice (negative sampling) when you only have one class of real-world data.

In [5]:
"""
STEP 2: Generate synthetic "suspicious" entries to pair with our genuine data.

We don't have real counterfeit examples, so we simulate the 3 most common
real-world counterfeiting patterns:
  1. Typo/character-swap attacks on the product name
  2. Tampered NRN (registration number) codes
  3. Completely fabricated product names

This gives our classifier both classes (genuine=1, suspicious=0) to learn from.
"""
import csv
import random
import re

random.seed(42)  # makes results reproducible - same output every run

def typo_name(name):
    """Simulate a typo: swap two adjacent letters, or drop one letter."""
    if len(name) < 4:
        return name + "x"
    i = random.randint(1, len(name) - 2)
    chars = list(name)
    action = random.choice(["swap", "drop", "double"])
    if action == "swap":
        chars[i], chars[i + 1] = chars[i + 1], chars[i]
    elif action == "drop":
        del chars[i]
    else:
        chars.insert(i, chars[i])
    return "".join(chars)

def tamper_nrn(nrn):
    """Simulate a tampered NRN: change one digit."""
    if not nrn:
        return "A0-000000"
    chars = list(nrn)
    digit_positions = [i for i, c in enumerate(chars) if c.isdigit()]
    if digit_positions:
        pos = random.choice(digit_positions)
        chars[pos] = random.choice("0123456789")
    return "".join(chars)

FAKE_PRODUCT_NAMES = [
    "Super Cure Tablet", "MiracleVax Injection", "PowerHeal Capsule",
    "QuickFix Antibiotic", "MaxStrength Syrup", "InstaHeal Suspension",
    "ForteMed Tablet", "BioShield Vaccine", "PureLife Injection",
    "RapidCure Solution", "VitaBoost Capsule", "TrustMed Tablet",
    "GenCure Injection", "HealFast Syrup", "MedPlus Extra Tablet",
]

def generate_fakes(genuine_rows, n_per_genuine=1):
    fakes = []
    for row in genuine_rows:
        # Pattern 1: typo in a real product's name, kept with a fake-ish NRN
        fakes.append({
            "product_name": typo_name(row["product_name"]),
            "nrn": row["nrn"],  # NRN copied correctly - name is the giveaway
            "label": 0,
            "fake_type": "name_typo",
        })
        # Pattern 2: correct name, tampered NRN
        fakes.append({
            "product_name": row["product_name"],
            "nrn": tamper_nrn(row["nrn"]),
            "label": 0,
            "fake_type": "nrn_tamper",
        })

    # Pattern 3: fully fabricated products with plausible-looking fake NRNs
    for name in FAKE_PRODUCT_NAMES:
        fake_nrn = f"A{random.randint(1,9)}-{random.randint(100000,999999)}"
        fakes.append({
            "product_name": name,
            "nrn": fake_nrn,
            "label": 0,
            "fake_type": "fabricated",
        })
    return fakes


if __name__ == "__main__":
    genuine_rows = list(csv.DictReader(open("data/nafdac_products_clean.csv", encoding="utf-8")))

    fakes = generate_fakes(genuine_rows)

    # Build the full labeled training set: genuine (label=1) + fake (label=0)
    training_rows = []
    for row in genuine_rows:
        training_rows.append({
            "product_name": row["product_name"],
            "nrn": row["nrn"],
            "label": 1,
            "fake_type": "genuine",
        })
    training_rows += fakes

    random.shuffle(training_rows)

    out_path = "data/training_data.csv"
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["product_name", "nrn", "label", "fake_type"])
        writer.writeheader()
        writer.writerows(training_rows)

    n_genuine = sum(1 for r in training_rows if r["label"] == 1)
    n_fake = sum(1 for r in training_rows if r["label"] == 0)
    print(f"Training set built -> {out_path}")
    print(f"  Genuine (label=1): {n_genuine}")
    print(f"  Suspicious (label=0): {n_fake}")
    print(f"  Total: {len(training_rows)}")


Training set built -> data/training_data.csv
  Genuine (label=1): 235
  Suspicious (label=0): 485
  Total: 720


## Step 3: Feature engineering
Convert text into numeric signals: name similarity to the closest genuine product, whether the NRN exists, whether the NRN format is valid, NRN similarity, and whether a high-similarity name is paired with the wrong NRN.

In [6]:
"""
STEP 3: Feature engineering.

A machine learning model can't read "Amoxicilin" and just know it's wrong -
it needs NUMBERS that capture the *pattern* of what looks suspicious.

For every product entry we're checking, we compute 5 signals by comparing
it against our genuine NAFDAC database:

1. name_similarity   - how close is this name to the CLOSEST genuine name?
                        (1.0 = exact match, 0.0 = nothing like it)
2. nrn_exact_match    - is this exact NRN code in our genuine database? (1/0)
3. nrn_format_valid   - does the NRN follow NAFDAC's pattern, e.g. "A6-100070"? (1/0)
4. nrn_similarity     - how close is the NRN to the closest genuine NRN?
5. name_nrn_mismatch  - does this name+NRN combo exist together for real?
                        (catches: real name + wrong NRN, or vice versa)

This is the "secret sauce" - a typo'd name will have HIGH name_similarity
(close but not perfect) while a fabricated product will have LOW similarity
to everything. The model learns to tell these apart.
"""
import csv
import re
import difflib

def load_genuine_db(path="data/nafdac_products_clean.csv"):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    names = [r["product_name"] for r in rows]
    nrns = [r["nrn"] for r in rows if r["nrn"]]
    name_to_nrn = {r["product_name"]: r["nrn"] for r in rows}
    return names, nrns, name_to_nrn

NRN_PATTERN = re.compile(r"^[A-Za-z0-9]{1,3}-\d{2,7}$")

def closest_match_score(query, candidates):
    """difflib gives a 0-1 similarity ratio - like a simple fuzzy-match score."""
    if not candidates:
        return 0.0, None
    best_score, best_match = 0.0, None
    for c in candidates:
        score = difflib.SequenceMatcher(None, query.lower(), c.lower()).ratio()
        if score > best_score:
            best_score, best_match = score, c
    return best_score, best_match

def extract_features(product_name, nrn, names_db, nrns_db, name_to_nrn):
    name_sim, closest_name = closest_match_score(product_name, names_db)
    nrn_exact = 1 if nrn in nrns_db else 0
    nrn_valid_format = 1 if nrn and NRN_PATTERN.match(nrn) else 0
    nrn_sim, _ = closest_match_score(nrn or "", nrns_db)

    # Does the closest-matching name actually pair with THIS nrn in real life?
    expected_nrn = name_to_nrn.get(closest_name, "")
    name_nrn_mismatch = 1 if (name_sim > 0.85 and nrn != expected_nrn) else 0

    return {
        "name_similarity": round(name_sim, 4),
        "nrn_exact_match": nrn_exact,
        "nrn_format_valid": nrn_valid_format,
        "nrn_similarity": round(nrn_sim, 4),
        "name_nrn_mismatch": name_nrn_mismatch,
    }


if __name__ == "__main__":
    names_db, nrns_db, name_to_nrn = load_genuine_db()

    training_rows = list(csv.DictReader(open("data/training_data.csv", encoding="utf-8")))

    feature_rows = []
    for row in training_rows:
        feats = extract_features(row["product_name"], row["nrn"], names_db, nrns_db, name_to_nrn)
        feats["label"] = row["label"]
        feats["product_name"] = row["product_name"]
        feature_rows.append(feats)

    out_path = "data/features.csv"
    fieldnames = ["product_name", "name_similarity", "nrn_exact_match",
                  "nrn_format_valid", "nrn_similarity", "name_nrn_mismatch", "label"]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(feature_rows)

    print(f"Features computed for {len(feature_rows)} rows -> {out_path}")
    print("\nSample (genuine vs suspicious side by side):")
    for r in feature_rows[:4]:
        print(f"  label={r['label']}  name_sim={r['name_similarity']:.2f}  "
              f"nrn_exact={r['nrn_exact_match']}  nrn_valid={r['nrn_format_valid']}  "
              f"| {r['product_name'][:40]}")


Features computed for 720 rows -> data/features.csv

Sample (genuine vs suspicious side by side):
  label=0  name_sim=1.00  nrn_exact=0  nrn_valid=1  | Accesslife Tramadol Capsules
  label=0  name_sim=0.97  nrn_exact=1  nrn_valid=1  | Lisin10 Tablets
  label=0  name_sim=1.00  nrn_exact=0  nrn_valid=1  | Actrapid FlexPen 100 IU
  label=1  name_sim=1.00  nrn_exact=1  nrn_valid=1  | Apitol Caplets Caplet Cyproheptadine Hyd


## Step 4: Train & evaluate the model
Random Forest classifier, trained on 80% of the data and evaluated on the held-out 20% it has never seen - this is what makes the evaluation trustworthy.

In [7]:
"""
STEP 4: Train a classifier and evaluate it properly.

We use a Random Forest - it builds many small decision trees (e.g. "IF
name_similarity < 0.95 AND nrn_exact_match == 0 THEN suspicious") and
averages their votes. It's a great choice for beginners because:
  - It handles our small feature set well
  - It's hard to badly misconfigure
  - It tells us which features actually mattered (interpretable)

CRITICAL RULE: we NEVER evaluate a model on the same data it trained on -
that's like grading a student's exam using the answer key they copied from.
We split into train/test sets so the model is judged on examples it has
never seen.
"""
import csv
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

FEATURE_COLS = ["name_similarity", "nrn_exact_match", "nrn_format_valid",
                 "nrn_similarity", "name_nrn_mismatch"]

df = pd.read_csv("data/features.csv")
X = df[FEATURE_COLS]
y = df["label"]

# 80% train, 20% test. stratify=y keeps the genuine/fake ratio balanced
# in both splits so the test set is a fair, representative sample.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,   # 200 decision trees voting together
    max_depth=6,        # keeps trees simple - avoids memorizing noise
    random_state=42,
)
model.fit(X_train, y_train)

# --- Evaluate on the held-out TEST set (data the model never saw) ---
y_pred = model.predict(X_test)

print("=" * 55)
print("EVALUATION RESULTS (on unseen test data)")
print("=" * 55)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}  "
      "(of predicted-genuine, how many really are genuine)")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}  "
      "(of actually-genuine, how many we correctly caught)")
print(f"F1 Score:  {f1_score(y_test, y_pred):.3f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"                 Predicted Fake   Predicted Genuine")
print(f"  Actual Fake         {cm[0][0]:<15} {cm[0][1]}")
print(f"  Actual Genuine      {cm[1][0]:<15} {cm[1][1]}")

print("\nFull classification report:")
print(classification_report(y_test, y_pred, target_names=["Suspicious", "Genuine"]))

print("Feature importance (what the model relies on most):")
importances = sorted(zip(FEATURE_COLS, model.feature_importances_),
                      key=lambda x: -x[1])
for feat, imp in importances:
    bar = "#" * int(imp * 50)
    print(f"  {feat:<20} {imp:.3f} {bar}")

# Save the trained model to disk so the app can load it instantly
joblib.dump(model, "data/fake_drug_model.joblib")
print("\nModel saved -> data/fake_drug_model.joblib")

EVALUATION RESULTS (on unseen test data)
Accuracy:  0.972
Precision: 0.922  (of predicted-genuine, how many really are genuine)
Recall:    1.000  (of actually-genuine, how many we correctly caught)
F1 Score:  0.959

Confusion Matrix:
                 Predicted Fake   Predicted Genuine
  Actual Fake         93              4
  Actual Genuine      0               47

Full classification report:
              precision    recall  f1-score   support

  Suspicious       1.00      0.96      0.98        97
     Genuine       0.92      1.00      0.96        47

    accuracy                           0.97       144
   macro avg       0.96      0.98      0.97       144
weighted avg       0.97      0.97      0.97       144

Feature importance (what the model relies on most):
  name_similarity      0.585 #############################
  name_nrn_mismatch    0.225 ###########
  nrn_exact_match      0.100 #####
  nrn_similarity       0.090 ####
  nrn_format_valid     0.000 

Model saved -> data/fake_

## Step 5: The checker app
Ties it together: type a product name + NRN, get a verdict with a plain-English explanation.

In [8]:

def check_product(product_name, nrn, model, names_db, nrns_db, name_to_nrn):
    feats = extract_features(product_name, nrn, names_db, nrns_db, name_to_nrn)
    X = pd.DataFrame([feats])[FEATURE_COLS]
    prediction = model.predict(X)[0]
    confidence = model.predict_proba(X)[0][prediction]
    verdict = "GENUINE" if prediction == 1 else "SUSPICIOUS"

    closest_name_score, closest_name = closest_match_score(product_name, names_db)
    reasons = []
    if feats["nrn_exact_match"] == 1:
        reasons.append(f"NRN '{nrn}' matches a registered NAFDAC product exactly.")
    else:
        reasons.append(f"NRN '{nrn}' was NOT found in the NAFDAC registry.")
    if closest_name_score > 0.99:
        reasons.append(f"Product name matches '{closest_name}' exactly.")
    elif closest_name_score > 0.80:
        reasons.append(f"Product name is suspiciously CLOSE to '{closest_name}' "
                        f"({closest_name_score:.0%} similar) but not exact - possible typo-squatting.")
    else:
        reasons.append(f"No close match found (closest: '{closest_name}', {closest_name_score:.0%} similar).")
    if feats["name_nrn_mismatch"] == 1:
        reasons.append("The name and NRN don't belong together in the real registry.")

    return {"verdict": verdict, "confidence": round(confidence * 100, 1), "reasons": reasons}

# Demo run
demo_cases = [
    ("Simulect", "A6-0405"),
    ("Simulcet", "A6-0405"),
    ("Ocrevus", "A6-999999"),
    ("Super Cure Tablet", "A9-123456"),
]
for name, nrn in demo_cases:
    result = check_product(name, nrn, model, names_db, nrns_db, name_to_nrn)
    print(f"\nChecking: '{name}'  |  NRN: {nrn}")
    print(f"  -> Verdict: {result['verdict']}  (confidence: {result['confidence']}%)")
    for r in result["reasons"]:
        print(f"     - {r}")



Checking: 'Simulect'  |  NRN: A6-0405
  -> Verdict: GENUINE  (confidence: 90.6%)
     - NRN 'A6-0405' matches a registered NAFDAC product exactly.
     - Product name matches 'Simulect' exactly.

Checking: 'Simulcet'  |  NRN: A6-0405
  -> Verdict: SUSPICIOUS  (confidence: 100.0%)
     - NRN 'A6-0405' matches a registered NAFDAC product exactly.
     - Product name is suspiciously CLOSE to 'Simulect' (88% similar) but not exact - possible typo-squatting.

Checking: 'Ocrevus'  |  NRN: A6-999999
  -> Verdict: SUSPICIOUS  (confidence: 100.0%)
     - NRN 'A6-999999' was NOT found in the NAFDAC registry.
     - Product name matches 'Ocrevus' exactly.
     - The name and NRN don't belong together in the real registry.

Checking: 'Super Cure Tablet'  |  NRN: A9-123456
  -> Verdict: SUSPICIOUS  (confidence: 100.0%)
     - NRN 'A9-123456' was NOT found in the NAFDAC registry.
     - No close match found (closest: 'Getryl 2 mg Tablets', 56% similar).


## Limitations & next steps
- Trained on the Vaccines/Biologics category (156 products) as a proof of concept; the same scraper extends to NAFDAC's full Drugs category (1,900+ products).
- 'Fake' examples are synthetic (typo/NRN-tamper/fabricated), not real seized counterfeit data - a production version should validate against real NAFDAC/WHO alert bulletins.
- Next: barcode/GS1 lookup support, Streamlit UI, expand to all 6 product categories.